# V7.7 — Decorrelated V7 bases on Kaggle GPU

V7.6 proved that adding more LightGBMs *with the same feature set* doesn't lift the LAD stack — the bases are too correlated.  This notebook trains **feature-decorrelated** V7 variants: each one drops a subset of features so its errors are structurally different from the existing v7/v71/v72.

Variants:
* `v77_nopromo`     — drop all promo-lifecycle features (let the base learn pure baseline demand)
* `v77_nosegment`   — drop categorical Канал/Бренд/Сегмент_ABC encodings
* `v77_recent`      — recency-weighted training (γ=1.2 per month, last 24 months)
* `v77_long`        — only pairs with ≥12 historical observations (long-history regime)
* `v77_quantile60`  — quantile=0.6 (asymmetric upward tilt)

Each saves `preds_v77_<variant>_{val,test}.csv` for the LAD stacker pool.

In [ ]:
import shutil, subprocess, sys
if shutil.which('nvidia-smi'):
    print(subprocess.check_output(['nvidia-smi', '-L']).decode())
print(sys.version)

In [ ]:
!pip install -q lightgbm==4.6.0 joblib pandas pyarrow scikit-learn
import os, shutil, sys
INPUT_DIR = '/kaggle/input'; REPO_DIR = '/kaggle/working/repo'
os.makedirs(REPO_DIR, exist_ok=True)

candidates = [d for d in os.listdir(INPUT_DIR) if os.path.isdir(f'{INPUT_DIR}/{d}')]
DATASET = next((d for d in candidates if 'bpm' in d.lower()), candidates[0])
DS_DIR = f'{INPUT_DIR}/{DATASET}'

def _locate(name):
    if os.path.isdir(f'{DS_DIR}/{name}'): return f'{DS_DIR}/{name}'
    for root, dirs, _ in os.walk(DS_DIR):
        if name in dirs: return os.path.join(root, name)
    return None
for sub in ('src', 'scripts'):
    loc = _locate(sub); assert loc, f'missing {sub}/'
    shutil.copytree(loc, f'{REPO_DIR}/{sub}', dirs_exist_ok=True)

os.makedirs(f'{REPO_DIR}/output', exist_ok=True)
def _find_file(name):
    for root, _, files in os.walk(DS_DIR):
        if name in files: return os.path.join(root, name)
    return None
for fn in ('abt_v7_cached.parquet',):
    found = _find_file(fn)
    if found: shutil.copy(found, f'{REPO_DIR}/output/{fn}'); print('copied', fn)
    else: raise FileNotFoundError(fn)
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
print('ready')

In [ ]:
import numpy as np, pandas as pd, time, json
from src.evaluation import split_train_val_test
from src.model_v2 import (
    TwoStageForecaster, encode_categoricals,
    filter_active_pairs, get_feature_columns_v2,
)

abt = pd.read_parquet('output/abt_v7_cached.parquet').pipe(encode_categoricals)
feats_full = [c for c in get_feature_columns_v2(abt) if c != 'target_qty_imputed']
print(f'ABT: {len(abt)} rows  |  full features: {len(feats_full)}')

df_train, df_val, df_test = split_train_val_test(abt)
active = filter_active_pairs(df_train)
keys = active[['Партнер', 'Артикул']].drop_duplicates()
df_val = df_val.merge(keys, on=['Партнер', 'Артикул'], how='inner')
df_test = df_test.merge(keys, on=['Партнер', 'Артикул'], how='inner')
print(f'train={len(active)}  val={len(df_val)}  test={len(df_test)}')

KEY = ['Период', 'Партнер', 'Артикул']

def save_preds(df, p, path):
    out = df[KEY + ['target_qty']].copy()
    out['prediction'] = np.clip(p, 0, None)
    out.to_csv(path, index=False)

def metric(y, p):
    wape = float(np.abs(y - p).sum() / max(float(y.sum()), 1.0))
    bias = float((p.sum() - y.sum()) / max(float(y.sum()), 1.0) * 100.0)
    return wape, bias


In [ ]:
# Feature-decorrelated subsets.
promo_kw = ('promo_', 'months_since_last_promo', 'months_until_next_promo',
            'post_promo_depletion_flag', 'sku_promo_sensitivity')
categ_kw = ('Канал', 'Бренд', 'Сегмент_ABC', 'Тип_соглашения')

feats_nopromo = [c for c in feats_full if not any(k in c for k in promo_kw)]
feats_nosegment = [c for c in feats_full if not any(k in c for k in categ_kw)]

print('full features:', len(feats_full))
print('nopromo features:', len(feats_nopromo))
print('nosegment features:', len(feats_nosegment))

In [ ]:
# Build recency-weighted training subset (last 24 months only, geometric weight).
import pandas as pd

active['__per'] = active['Период'].dt.to_timestamp() if isinstance(active['Период'].dtype, pd.PeriodDtype) else pd.to_datetime(active['Период'])
max_per = active['__per'].max()
active['__age'] = ((max_per.year - active['__per'].dt.year) * 12 + (max_per.month - active['__per'].dt.month))

active_recent = active[active['__age'] <= 36].copy()
# Geometric weight γ=1.05 → most recent month gets ~1, 36 months ago gets ~0.17
active_recent['__w'] = (1.0 / 1.05) ** active_recent['__age']
print(f'recent training rows: {len(active_recent)}')

# Long-history pairs (≥18 months observed in train).
obs_count = (active.groupby(['Партнер', 'Артикул']).size().rename('n_obs').reset_index())
long_pairs = obs_count[obs_count['n_obs'] >= 18][['Партнер', 'Артикул']]
active_long = active.merge(long_pairs, on=['Партнер', 'Артикул'], how='inner')
print(f'long-history rows: {len(active_long)} ({len(long_pairs)} pairs)')

In [ ]:
VARIANTS = []

def base_clf(): return {'num_leaves': 127, 'learning_rate': 0.05, 'min_child_samples': 30, 'verbose': -1}
def base_reg(): return {'num_leaves': 127, 'learning_rate': 0.05, 'min_child_samples': 20, 'feature_fraction': 0.85, 'bagging_fraction': 0.85, 'bagging_freq': 5, 'verbose': -1}

VARIANTS.append(dict(name='nopromo',     train=active,        feats=feats_nopromo,   reg={**base_reg(), 'objective': 'tweedie', 'tweedie_variance_power': 1.3}, clf=base_clf(), weight_col=None))
VARIANTS.append(dict(name='nosegment',   train=active,        feats=feats_nosegment, reg={**base_reg(), 'objective': 'regression_l1'},                            clf=base_clf(), weight_col=None))
VARIANTS.append(dict(name='recent',      train=active_recent, feats=feats_full,      reg={**base_reg(), 'objective': 'regression_l1'},                            clf=base_clf(), weight_col='__w'))
VARIANTS.append(dict(name='long',        train=active_long,   feats=feats_full,      reg={**base_reg(), 'objective': 'tweedie', 'tweedie_variance_power': 1.3}, clf=base_clf(), weight_col=None))
VARIANTS.append(dict(name='quantile60',  train=active,        feats=feats_full,      reg={**base_reg(), 'objective': 'quantile', 'alpha': 0.6},                  clf=base_clf(), weight_col=None))

metrics_rows = []

for v in VARIANTS:
    print(f'\n======== training v77_{v["name"]} (objective={v["reg"]["objective"]}, feats={len(v["feats"])}, train_rows={len(v["train"])}) ========')
    model = TwoStageForecaster(
        clf_params=v['clf'],
        reg_params=v['reg'],
        reg_objective='',
        target_col='target_qty_imputed',
    )
    t0 = time.time()
    sw = v['train'][v['weight_col']].to_numpy() if v.get('weight_col') and v['weight_col'] in v['train'].columns else None
    model.fit(v['train'], df_val, v['feats'], num_boost_round=1200, early_stopping=50, sample_weight_train=sw)
    dt = time.time() - t0
    p_val  = model.predict(df_val)
    p_test = model.predict(df_test)
    save_preds(df_val,  p_val,  f'/kaggle/working/preds_v77_{v["name"]}_val.csv')
    save_preds(df_test, p_test, f'/kaggle/working/preds_v77_{v["name"]}_test.csv')
    vw, vb = metric(df_val['target_qty'].to_numpy(),  p_val)
    tw, tb = metric(df_test['target_qty'].to_numpy(), p_test)
    print(f'  fit in {dt:.1f}s  |  val WAPE={vw:.4f} bias={vb:+.2f}  |  test WAPE={tw:.4f} bias={tb:+.2f}')
    metrics_rows.append(dict(name=v['name'], fit_sec=round(dt,1), val_WAPE=vw, val_bias_pct=vb, test_WAPE=tw, test_bias_pct=tb))

pd.DataFrame(metrics_rows).to_csv('/kaggle/working/v77_decorr_metrics.csv', index=False)
print(); print(pd.DataFrame(metrics_rows).to_string(index=False))

In [ ]:
import os, zipfile
with zipfile.ZipFile('/kaggle/working/v77_decorr_bundle.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir('/kaggle/working'):
        if f.startswith('preds_v77_') or f.startswith('v77_decorr'):
            z.write(f'/kaggle/working/{f}', f); print('zipped', f)